In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from onnxruntime.transformers.models.stable_diffusion.benchmark import example_prompts

"""
Prompt Template:通过模板管理大模型的输入
    在应用开发中，固定的提示词限制了模型饿灵活性和适用范围，所以，Prompt Template是
    一个模板化的字符串，你可以将变量插入到模板中，从而创建出不同的提示。调用时：
        以字典作为输入，其中每个键代表要填充的提示模板中的变量
        输出一个promptValue，这个promptValue可以传递给LLM或ChatModel，并且还可以转为字符串或消息列表
    不同类型的提示模板
        PromptTemplate:LLM提示模板，用于生成字符串提示，它使用Python的字符串来模板提示
        ChatPromptTemplate：聊天提示模板，用于组合各种角色的消息模板，传入聊天模型
        XxxMessagePromptTemplate:消息模板提示词模板
        FewShotPromptTemplate:样本提示词模板，通过示例来教模型如何回答
        PipelinePromptTemplate:管道提示词模板，用于把几个提示词组合在一起使用
        自定义模板：允许基于其它模板类来定制自己的提示词模板

"""

In [5]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, MessagesPlaceholder

# 1.PromptTemplate如何获取实例

# 使用构造方法的模式
prompt_template = PromptTemplate(template="你是一个{role},你的名字叫{name}",input_variables=['role','name'])

prompt = prompt_template.format(role='人工智能专家',name='小智')

print(prompt)

# 2.from_template()
prompt_template = PromptTemplate.from_template(template="你是一个{role},你的名字叫{name}")

prompt = prompt_template.format(role='人工智能专家',name='小智')

print(prompt)


你是一个人工智能专家,你的名字叫小智
你是一个人工智能专家,你的名字叫小智


In [2]:
from langchain_core.prompts import PromptTemplate

# 2.两种特殊结构的使用（部分提示词模板的使用，组和提示词的使用）
# 部分提示词模板
# 使用partial_variables设置
# prompt_template = PromptTemplate.from_template(
#     template="请你评价{product}的优缺点，包括{aspcet1}和{aspcet2}",
#     partial_variables={"aspcet1":'电池续航'}
# )
#
# prompt = prompt_template.format(product='智能手机',aspcet2='拍照质量')
#
# print(prompt)

# 调用方法
prompt_template = PromptTemplate.from_template(
    template="请你评价{product}的优缺点，包括{aspcet1}和{aspcet2}"
).partial(aspcet1='电池续航')

# partial()调用完以后，不会对调用者这个模板对象产生影响，而其返回值是一个新的模板
# prompt_template = prompt_template.partial(aspcet1='电池续航')

prompt = prompt_template.format(product='智能手机',aspcet2='拍照质量')

print(prompt)

# 组合提示词的使用
"""
LangChain 的 PromptTemplate 重载了 + 运算符，允许你将：
    多个 PromptTemplate 对象
    字符串（会自动转换为 PromptTemplate）
    以及它们的组合
"""
template = (
    PromptTemplate.from_template(template="Tell me a joke about {topic}")
    + ",make it funny"
    + "\n\nand in {language}"
)

prompt = template.format(topic='sports',language = 'spanish')
print(prompt)

请你评价智能手机的优缺点，包括电池续航和拍照质量
Tell me a joke about sports,make it funny

and in spanish


In [7]:
 # 3.给变量赋值的两种方法 format()/invoke()

# format():参数部分:给变量赋值，返回值：str类型
# invoke():参数部分:使用的是字典，返回值:PromptValue

prompt_template = PromptTemplate.from_template(
    template="请你评价{product}的优缺点，包括{aspcet1}和{aspcet2}"
).partial(aspcet1='电池续航')

# prompt = prompt_template.format(product='智能手机',aspcet2='拍照质量')
prompt = prompt_template.invoke(input={'product':'智能手机','aspcet2':'拍照质量'})
print(prompt)
print(type(prompt))

text='请你评价智能手机的优缺点，包括电池续航和拍照质量'
<class 'langchain_core.prompt_values.StringPromptValue'>


In [16]:

from langchain_openai import ChatOpenAI

prompt_template = PromptTemplate.from_template(
    template="请你评价{product}的优缺点，包括{aspcet1}和{aspcet2}"
).partial(aspcet1='电池续航')

prompt = prompt_template.invoke(input={'product':'智能手机','aspcet2':'拍照质量'})




llm = ChatOpenAI(
    model_name="qwen3.6-plus",# 默认使用的是gpt3.5
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=""
)

# 调用模型
resp = llm.invoke(prompt)

# 查看响应文本
print(resp.content)

智能手机已成为现代人不可或缺的数字终端，其优缺点往往是一体两面的技术权衡。以下从整体维度出发，并重点针对**电池续航**与**拍照质量**进行专项评价，力求客观、结构化。

---
### 一、智能手机的整体优点
1. **高度集成与便携性**：将通信、导航、支付、娱乐、办公等功能浓缩于掌心设备，实现“一机走天下”。
2. **生态互联与智能化**：与可穿戴设备、智能家居、云端服务无缝协同，AI语音助手、场景自动化大幅提升效率。
3. **技术迭代迅速**：芯片制程、屏幕素质、网络协议（5G/5.5G）、系统优化持续升级，体验边界不断拓展。
4. **降低专业门槛**：摄影、剪辑、设计、编程等原本需要专业设备的技能，如今可通过手机+App实现平民化。

### 二、智能手机的整体缺点
1. **注意力碎片化与心理依赖**：高频通知、算法推荐易导致信息过载、睡眠干扰与数字成瘾。
2. **隐私与安全风险**：位置追踪、后台权限滥用、数据泄露事件频发，用户让渡部分隐私换取便利。
3. **维修困难与电子浪费**：一体化设计、零件加密、系统锁定使自主维修成本高昂，2-4年换代周期加剧资源消耗。
4. **场景局限性**：极端环境（高寒/高温/水下/强冲击）或断网断电状态下，功能大幅受限。

---
### 三、专项评价：电池续航
#### ✅ 优点
- **容量与快充普及**：主流机型普遍配备4500~6000mAh电池，配合30W~240W有线快充、15W~50W无线快充，30分钟内可回血50%~80%。
- **系统级能效优化**：自适应刷新率（1~120Hz动态调节）、AI后台调度、低功耗协处理器、深色模式等显著延长实际使用时间。
- **充电生态完善**：PD/PPS协议兼容度高，车载/公共场所快充覆盖广，反向充电可应急为其他设备供电。

#### ❌ 缺点
- **物理瓶颈难突破**：锂离子电池能量密度已接近理论极限，轻薄化与长续航存在天然矛盾。
- **高负载耗电剧增**：5G常驻、高刷屏、AI大模型本地推理、高性能游戏等场景下，电池衰减速度成倍增加。
- **老化与热管理问题**：循环充放电2~3年后容量普遍衰减15%~25%；快充与高性能并发易引发发热，加速电池衰减并触发降频保护。
- **“续航焦虑”未根本消除**：中度以上用户仍需依赖充电宝或碎片化补电

In [18]:
# ChatPromptTemplate
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate


"""
ChatPromptTemplate是创建聊天消息列表的提示模板，它比普通PromptTemplate更适合处理多角色，多轮次的对话场景
特点：
    支持System/Human/AI等不同角色的消息模板
    对话历史维护
参数类型：列表参数格式是突破了类型（rol:str content:str组合最常用）
元素的格式为
    (role:str | type,content:str | list[dict] | list[object])
    其中role是：字符串（如：'system','human','ai）
"""

# 1.实例化的方式（两种方式：使用构造方法，from_message()）
# chat_prompt_template = ChatPromptTemplate(
#     messages=[
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         ('human','我的问题是{question}')
#     ],
#     input_variables=['name','question']
# )
#
# resp = chat_prompt_template.invoke(input={'name':'小智','question':'1 + 2 * 3 = ?'})
# print(resp)
#
# chat_prompt_template = ChatPromptTemplate.from_messages(
#     messages=[
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         ('human','我的问题是{question}')
#     ]
# )
#
# resp = chat_prompt_template.invoke(input={'name':'小智','question':'1 + 2 * 3 = ?'})
# print(resp)
# print(type(resp))

# 2.调用提示词模板的几种方法：invoke() format() format_message() format_prompt
# chat_prompt_template = ChatPromptTemplate(
#     messages=[
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         ('human','我的问题是{question}')
#     ],
#     input_variables=['name','question']
# )
#
# resp = chat_prompt_template.invoke(input={'name':'小智','question':'1 + 2 * 3 = ?'})
# print(resp)
# print(type(resp))
#
# chat_prompt_template = ChatPromptTemplate(
#     messages=[
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         ('human','我的问题是{question}')
#     ],
#     input_variables=['name','question']
# )
#
# resp = chat_prompt_template.format(name='小智',question='1 + 2 * 3 = ?')
# print(resp)
# print(type(resp))

#
#
# chat_prompt_template = ChatPromptTemplate(
#     messages=[
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         ('human','我的问题是{question}')
#     ],
#     input_variables=['name','question']
# )
#
# resp = chat_prompt_template.format_messages(name='小智',question='1 + 2 * 3 = ?')
# print(resp)
# print(type(resp))
# #
#
#
#
# chat_prompt_template = ChatPromptTemplate(
#     messages=[
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         ('human','我的问题是{question}')
#     ],
#     input_variables=['name','question']
# )
#
# resp = chat_prompt_template.format_prompt(name='小智',question='1 + 2 * 3 = ?')
# print(resp)
# print(type(resp))
# # 转换成list[message]
# print(resp.to_messages())
# # 转换成字符串
# print(resp.to_string())


# 3.更丰富的实例化参数类型
"""
本质：不管使用构造方法，还是使用from_message()来创建chatPromptTemplate的实例，本质上来讲，传入的都是消息构成的列表

从调用上来讲，我们看到，不管使用构造方法，还是使用from_message()，message的参数的类型都是列表，但是列表的元素的类型是多样的，元素可以是：
    字符串类型，字典类型，消息类型，元组构成的列表（最常用，最基础，最简单），Chat提示词模板类型，消息提示词模板类型
"""


# 元组构成的列表
# chat_prompt_template = ChatPromptTemplate(
#     messages=[
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         ('human','我的问题是{question}')
#     ],
#     input_variables=['name','question']
# )
#
# resp = chat_prompt_template.format_messages(name='小智',question='1 + 2 * 3 = ?')
# print(resp)
# print(type(resp))

# 字符串
# chat_prompt_template = ChatPromptTemplate.from_messages([
#     '我的问题是{question}' # 默认的角色是：human
# ])
#
# resp = chat_prompt_template.invoke({"question":'1 + 2 * 3 = ?'})
# print(resp)
# print(type(resp))


# 字典类型
# chat_prompt_template = ChatPromptTemplate.from_messages([
#     {
#         'role':'system',
#         'content':'我是一个人工智能助手，我的名字叫{name}'
#     },
#     {
#         'role':'human',
#         'content':'我的问题是{question}'
#     },
# ])
#
# resp = chat_prompt_template.invoke({"name":"小智","question":'1 + 2 * 3 = ?'})
# print(resp)
# print(type(resp))


# 消息类型
#
# chat_prompt_template = ChatPromptTemplate.from_messages([
#     SystemMessage(content='我是一个人工智能助手，我的名字叫{name}'),
#     HumanMessage(content='我的问题是{question}')
# ])
#
# resp = chat_prompt_template.invoke({"name":"小智","question":'1 + 2 * 3 = ?'})
# print(resp)
# print(type(resp))


# chat提示词模板类型
#
# chat_prompt_template_1 = ChatPromptTemplate.from_messages([
#     ('system','你是一个AI助手，你的名字叫做{name}')
# ])
#
# chat_prompt_template_2 = ChatPromptTemplate.from_messages([
#     ('human','我的问题是{question}')
# ])
#
#
# chat_prompt_template = ChatPromptTemplate.from_messages([
#     chat_prompt_template_1,
#     chat_prompt_template_2
# ])
#
# resp = chat_prompt_template.invoke({"name":"小智","question":'1 + 2 * 3 = ?'})
# print(resp)
# print(type(resp))



# 消息提示词模板类型
#
#
# system_template = '你是一个专家{role}'
# system_template_prompt = SystemMessagePromptTemplate.from_template(system_template)
#
#
# human_template = '给我解释{concept}，用浅显易懂的语言'
# human_template_prompt = SystemMessagePromptTemplate.from_template(human_template)
#
# chat_prompt_template = ChatPromptTemplate.from_messages([
#     system_template,human_template
# ])
#
#
# formatted_message = chat_prompt_template.format_messages(
#     role='物理学家',
#     concept='相对论'
# )
#
# print(formatted_message)
# print(type(formatted_message))



messages=[SystemMessage(content='我是一个人工智能助手，我的名字叫{name}', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是{question}', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


In [20]:
from langchain_openai import ChatOpenAI

# 4.结合LLM

chat_prompt_template_1 = ChatPromptTemplate.from_messages([
    ('system','你是一个AI助手，你的名字叫做{name}')
])

chat_prompt_template_2 = ChatPromptTemplate.from_messages([
    ('human','我的问题是{question}')
])


chat_prompt_template = ChatPromptTemplate.from_messages([
    chat_prompt_template_1,
    chat_prompt_template_2
])

resp = chat_prompt_template.invoke({"name":"小智","question":'1 + 2 * 3 = ?'})


llm = ChatOpenAI(
    model_name="qwen3.7-plus",# 默认使用的是gpt3.5
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=""
)

resp = llm.invoke(resp)

print(resp.content)

1 + 2 * 3 = 7。

根据数学中的四则运算优先级规则，需要先计算乘法，再计算加法：
1. 先算乘法：2 * 3 = 6
2. 再算加法：1 + 6 = 7

所以最终结果是 7。


In [24]:
# 5.插入消息列表:MessagePlaceholder
from langchain_core.prompts import MessagesPlaceholder
from  langchain_core.messages import AIMessage

"""
当你不确定消息提示模板使用什么角色,或者希望在格式化过程中插入消息列表时,该怎么办,这就需要MessagePlaceholder,
负责在特定的位置添加消息列表

使用场景:多轮多话系统存储历史消息以及Agent的中间步骤处理此功能非常有用
"""
#
# chat_prompt_template = ChatPromptTemplate(
#     [
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         MessagesPlaceholder(variable_name='msgs')
#     ]
# )
#
# chat_prompt_template.invoke({
#     'name':'小智',
#     'msgs':[HumanMessage(content='我是问题是 1 + 2 * 3 = ?')]
# })


#
# chat_prompt_template = ChatPromptTemplate(
#     [
#         ('system','你是一个AI助手，你的名字叫做{name}'),
#         MessagesPlaceholder(variable_name='msgs')
#     ]
# )
#
# chat_prompt_template.invoke({
#     'name':'小智',
#     'msgs':[HumanMessage(content='我是问题是 1 + 2 * 3 = ?'),AIMessage(content='1 + 2 * 3 = 7')]
# })




chat_prompt_template = ChatPromptTemplate(
    [
        ('system','你是一个AI助手，你的名字叫做{name}'),
        MessagesPlaceholder(variable_name='history'),
        ('human','{question}')
    ]
)

chat_prompt_template.invoke({
    'name':'小智',
    'history':[HumanMessage(content='我是问题是 1 + 2 * 3 = ?')
        ,AIMessage(content='1 + 2 * 3 = 7')],
    'question':'我刚才问题是什么?'
})


ChatPromptValue(messages=[SystemMessage(content='你是一个AI助手，你的名字叫做小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='我是问题是 1 + 2 * 3 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='1 + 2 * 3 = 7', additional_kwargs={}, response_metadata={}), HumanMessage(content='我刚才问题是什么?', additional_kwargs={}, response_metadata={})])

In [26]:
from langchain_openai import ChatOpenAI

chat_prompt_template = ChatPromptTemplate(
    [
        ('system','你是一个AI助手，你的名字叫做{name}'),
        MessagesPlaceholder(variable_name='history'),
        ('human','{question}')
    ]
)

value = chat_prompt_template.invoke({
    'name':'小智',
    'history':[HumanMessage(content='我的问题是 1 + 2 * 3 = ?')
        ,AIMessage(content='1 + 2 * 3 = 7')],
    'question':'我刚才问题是什么?'
})

llm = ChatOpenAI(
    model_name="qwen3.7-plus",# 默认使用的是gpt3.5
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=""
)

resp = llm.invoke(value)

print(resp.content)

你刚才的问题是：1 + 2 * 3 = ?


In [32]:

# 少量样本示例的提示词模板
"""
在构建prompt时,可以通过构建一个少量示例列表去进一步格式化prompt,这是一种简单但强大的指导生成的方式,在某些情况下可以显著提高模型性能

FewShotPromptTemplate:与promptTemplate一起使用
FewShotChatMessagePromptTemplate:与ChatPromptTemplate一起使用
Example selectors(示例选择器):
"""

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, MessagesPlaceholder

from langchain_openai import ChatOpenAI
from langchain_core.prompts import FewShotPromptTemplate


example_prompt = PromptTemplate.from_template(
    template="input:{input}\noutput:{output}",
)

examples=[
    {"input":"北京天气怎么样","output":"北京市"},
    {"input":"南京下雨吗","output":"南京市"},
    {"input":"武汉热吗","output":"武汉市"},
]

fewShotPromptTemplate = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="input:{input}\noutput:",# 声明在示例后面的提示词模板
    input_variables=["input"],

)

llm = ChatOpenAI(
    model_name="qwen3.7-plus",# 默认使用的是gpt3.5
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=""
)

resp = fewShotPromptTemplate.invoke({"input":"天津会下雨吗?"})


resp = llm.invoke(resp)

print(resp.content)


天津市
